In [2]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. إعداد الرابط والـ Headers
url = "https://www.transfermarkt.com/zamalek-sc/leistungsdaten/verein/664"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
  soup = BeautifulSoup(response.content, "html.parser")

  # 2. تحديد جدول اللعيبة
  table = soup.find("table", class_="items")
  tbody = table.find("tbody")
  rows = tbody.find_all("tr", class_=["odd", "even"])

  players_data = []

  for row in rows:
    cells = row.find_all("td", recursive=False)
    if len(cells) < 4:
      continue

    # رقم القميص
    shirt_div = row.find("div", class_="rn_nummer")
    shirt_num = shirt_div.text.strip() if shirt_div else None

    # الاسم ورابط البروفايل
    hauptlink = row.find("td", class_="hauptlink")
    if not hauptlink:
      continue

    name_anchor = hauptlink.find("span", class_="hide-for-small")
    if name_anchor:
      name_a = name_anchor.find("a")
      player_name = name_a.text.strip() if name_a else None
      player_href = name_a.get("href", "") if name_a else ""
    else:
      player_name = None
      player_href = ""

    # استخراج الـ ID من الرابط
    player_id_match = re.search(r"/spieler/(\d+)", player_href)
    player_id = player_id_match.group(1) if player_id_match else None

    # المركز
    inline_table = row.find("table", class_="inline-table")
    position = None
    if inline_table:
      pos_rows = inline_table.find_all("tr")
      if len(pos_rows) > 1:
        position = pos_rows[1].text.strip()

    # السن والجنسية
    age_text = cells[2].text.strip()
    age = int(age_text) if age_text.isdigit() else None

    flag_img = cells[3].find("img")
    nationality = flag_img.get("title") if flag_img else None

    # فحص المشاركات والأرقام
    row_text = row.get_text()
    is_not_active = (
        "Not in squad" in row_text
        or "Not used" in row_text
        or row.find("td", {"colspan": "10"})
    )

    if is_not_active:
      in_squad_val = cells[4].text.strip() if len(cells) > 4 else "0"
      in_squad = int(in_squad_val) if in_squad_val.isdigit() else 0
      appearances = 0
      goals = 0
      minutes = 0
    else:
      # In squad
      in_squad_val = cells[4].text.strip()
      in_squad = int(in_squad_val) if in_squad_val.isdigit() else 0

      # Appearances
      app_val = cells[5].text.strip()
      appearances = int(app_val) if app_val.isdigit() else 0

      # Goals (مع تنظيف المسافات وعلامة الشرطة)
      goals_raw = cells[6].text.strip().replace("\xa0", "").replace("-", "0")
      goals = int(goals_raw) if goals_raw.isdigit() else 0

      # Minutes played (سحب أي رقم من خانة الدقائق rechts)
      min_td = row.find("td", class_=lambda c: c and "rechts" in c)
      if min_td:
        min_cleaned = re.sub(r"[^\d]", "", min_td.text)
        minutes = int(min_cleaned) if min_cleaned else 0
      else:
        minutes = 0

    players_data.append({
        "player_id": player_id,
        "shirt_number": shirt_num,
        "player_name": player_name,
        "team": "Zamalek SC",
        "position": position,
        "age": age,
        "nationality": nationality,
        "in_squad": in_squad,
        "appearances": appearances,
        "goals": goals,
        "minutes_played": minutes,
    })

  # 3. حفظ الـ CSV وعرض النتيجة
  df_zamalek = pd.DataFrame(players_data)
  df_zamalek.to_csv("zamalek_players.csv", index=False, encoding="utf-8-sig")
  print(f"تم بنجاح! تم استخراج بيانات {len(df_zamalek)} لاعب.")
  display(df_zamalek.head(10))

else:
  print(f"فشل الطلب! كود الاستجابة: {response.status_code}")

تم بنجاح! تم استخراج بيانات 35 لاعب.


,player_id,shirt_number,player_name,team,position,age,nationality,in_squad,appearances,goals,minutes_played
0,583889,16,Mohamed Sobhi,Zamalek SC,Goalkeeper,27,Egypt,3,1,0,90
1,104951,37,El Mahdi Soliman,Zamalek SC,Goalkeeper,39,Egypt,6,6,0,540
2,184818,99,Mohamed Awad,Zamalek SC,Goalkeeper,34,Egypt,6,1,0,90
3,1274483,46,Mahmoud El Shenawy,Zamalek SC,Goalkeeper,22,Egypt,0,0,0,0
4,1254974,38,Omar Abdelaziz,Zamalek SC,Goalkeeper,18,Egypt,1,0,0,0
5,712192,24,Mohamed Ismail,Zamalek SC,Centre-Back,27,Egypt,8,8,1,720
6,840920,3,Mahmoud Bentayg,Zamalek SC,Left-Back,26,Morocco,7,6,0,524
7,465966,13,Ahmed Fatouh,Zamalek SC,Left-Back,28,Egypt,5,5,0,243
8,308526,28,Mahmoud El Wensh,Zamalek SC,Centre-Back,31,Egypt,8,8,0,720
9,580923,20,Mostafa El Zenary,Zamalek SC,Centre-Back,27,Egypt,8,2,0,26


In [3]:
import os
import random
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# قاموس الأندية الـ 20 مع روابط صفحات الإحصائيات (leistungsdaten)
TEAMS_CONFIG = {
    "Wadi Degla FC": {
        "slug": "wadi_degla",
        "url": (
            "https://www.transfermarkt.com/wadi-degla-fc/leistungsdaten/verein/24999"
        ),
    },
    "Modern Sport Club": {
        "slug": "modern_sport",
        "url": (
            "https://www.transfermarkt.com/modern-sport-fc/leistungsdaten/verein/32701"
        ),
    },
    "Bank El Ahly": {
        "slug": "bank_el_ahly",
        "url": (
            "https://www.transfermarkt.com/national-bank-egypt/leistungsdaten/verein/62448"
        ),
    },
    "Cleopatra FC": {
        "slug": "cleopatra_fc",
        "url": (
            "https://www.transfermarkt.com/ceramica-cleopatra-fc/leistungsdaten/verein/62325"
        ),
    },
    "Zed FC": {
        "slug": "zed_fc",
        "url": (
            "https://www.transfermarkt.com/zed-fc/leistungsdaten/verein/62324"
        ),
    },
    "Pyramids FC": {
        "slug": "pyramids_fc",
        "url": (
            "https://www.transfermarkt.com/pyramids-fc/leistungsdaten/verein/64019"
        ),
    },
    "Abo Qir Fertilizers": {
        "slug": "abo_qir_fertilizers",
        "url": (
            "https://www.transfermarkt.com/abu-qair-semad/leistungsdaten/verein/25333"
        ),
    },
    "Qanah FC": {
        "slug": "qanah_fc",
        "url": (
            "https://www.transfermarkt.com/el-qanah-fc/leistungsdaten/verein/12582"
        ),
    },
    "Smouha SC": {
        "slug": "smouha_sc",
        "url": (
            "https://www.transfermarkt.com/smouha-sc/leistungsdaten/verein/23387"
        ),
    },
    "El Gouna FC": {
        "slug": "el_gouna_fc",
        "url": (
            "https://www.transfermarkt.com/el-gouna-fc/leistungsdaten/verein/19875"
        ),
    },
    "Al Ahly FC": {
        "slug": "al_ahly",
        "url": (
            "https://www.transfermarkt.com/el-ahly-cairo/leistungsdaten/verein/7"
        ),
    },
    "Petrol Asyut": {
        "slug": "petrol_asyut",
        "url": (
            "https://www.transfermarkt.com/petrol-asyut/leistungsdaten/verein/12620"
        ),
    },
    "Ghazl El Mahalla": {
        "slug": "ghazl_el_mahalla",
        "url": (
            "https://www.transfermarkt.com/ghazl-el-mahalla/leistungsdaten/verein/12586"
        ),
    },
    "Suez Petrojet FC": {
        "slug": "petrojet_fc",
        "url": (
            "https://www.transfermarkt.com/petrojet/leistungsdaten/verein/12619"
        ),
    },
    "Tala'ea El Gaish": {
        "slug": "talaea_el_gaish",
        "url": (
            "https://www.transfermarkt.com/talaea-el-gaish/leistungsdaten/verein/12613"
        ),
    },
    "El Sharkia Enppi FC": {
        "slug": "enppi_fc",
        "url": (
            "https://www.transfermarkt.com/enppi-sc/leistungsdaten/verein/4156"
        ),
    },
    "El Masry SC": {
        "slug": "el_masry_sc",
        "url": (
            "https://www.transfermarkt.com/el-masry-sc/leistungsdaten/verein/2534"
        ),
    },
    "Ittihad Alexandria SC": {
        "slug": "ittihad_alexandria",
        "url": (
            "https://www.transfermarkt.com/ittihad-alexandria/leistungsdaten/verein/12588"
        ),
    },
    "El Mokawloon SC": {
        "slug": "el_mokawloon_sc",
        "url": (
            "https://www.transfermarkt.com/al-mokawloon-al-arab/leistungsdaten/verein/3067"
        ),
    },
    "Zamalek SC": {
        "slug": "zamalek_sc",
        "url": (
            "https://www.transfermarkt.com/zamalek-sc/leistungsdaten/verein/664"
        ),
    },
}

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def scrape_team_roster(team_display_name, team_slug, team_url):
  print(f"\n[+] جارٍ معالجة: {team_display_name}...")

  try:
    response = requests.get(team_url, headers=headers, timeout=15)
    if response.status_code != 200:
      print(f"[-] تعذر جلب البيانات. الرمز: {response.status_code}")
      return

    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table", class_="items")
    if not table:
      print(f"[-] لم يتم العثور على جدول اللاعبين لـ {team_display_name}")
      return

    tbody = table.find("tbody")
    rows = tbody.find_all("tr", class_=["odd", "even"])

    records = []

    for row in rows:
      cells = row.find_all("td", recursive=False)
      if len(cells) < 4:
        continue

      shirt_div = row.find("div", class_="rn_nummer")
      shirt_num = shirt_div.text.strip() if shirt_div else None

      hauptlink = row.find("td", class_="hauptlink")
      if not hauptlink:
        continue

      name_anchor = hauptlink.find("span", class_="hide-for-small")
      if name_anchor:
        name_a = name_anchor.find("a")
        player_name = name_a.text.strip() if name_a else None
        player_href = name_a.get("href", "") if name_a else ""
      else:
        player_name = None
        player_href = ""

      player_id_match = re.search(r"/spieler/(\d+)", player_href)
      player_id = player_id_match.group(1) if player_id_match else None

      inline_table = row.find("table", class_="inline-table")
      position = None
      if inline_table:
        pos_rows = inline_table.find_all("tr")
        if len(pos_rows) > 1:
          position = pos_rows[1].text.strip()

      age_text = cells[2].text.strip()
      age = int(age_text) if age_text.isdigit() else None

      flag_img = cells[3].find("img")
      nationality = flag_img.get("title") if flag_img else None

      row_text = row.get_text()
      is_inactive = (
          "Not in squad" in row_text
          or "Not used" in row_text
          or row.find("td", {"colspan": "10"})
      )

      if is_inactive:
        in_squad_val = cells[4].text.strip() if len(cells) > 4 else "0"
        in_squad = int(in_squad_val) if in_squad_val.isdigit() else 0
        appearances = 0
        goals = 0
        minutes = 0
      else:
        in_squad_val = cells[4].text.strip()
        in_squad = int(in_squad_val) if in_squad_val.isdigit() else 0

        app_val = cells[5].text.strip()
        appearances = int(app_val) if app_val.isdigit() else 0

        goals_raw = cells[6].text.strip().replace("\xa0", "").replace("-", "0")
        goals = int(goals_raw) if goals_raw.isdigit() else 0

        min_td = row.find("td", class_=lambda c: c and "rechts" in c)
        if min_td:
          min_cleaned = re.sub(r"[^\d]", "", min_td.text)
          minutes = int(min_cleaned) if min_cleaned else 0
        else:
          minutes = 0

      records.append({
          "player_id": player_id,
          "shirt_number": shirt_num,
          "player_name": player_name,
          "team": team_display_name,
          "position": position,
          "age": age,
          "nationality": nationality,
          "in_squad": in_squad,
          "appearances": appearances,
          "goals": goals,
          "minutes_played": minutes,
      })

    # حفظ ملف CSV منفصل للفريق
    filename = f"{team_slug}_players.csv"
    df = pd.DataFrame(records)
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"[✓] تم حفظ {len(df)} لاعب في: {filename}")

  except Exception as err:
    print(f"[-] خطأ أثناء معالجة {team_display_name}: {err}")


# تنفيذ حلقة السحب عبر جميع الفرق الـ 20
total_teams = len(TEAMS_CONFIG)
for idx, (display_name, info) in enumerate(TEAMS_CONFIG.items(), start=1):
  print(f"\n--- [{idx}/{total_teams}] ---")
  scrape_team_roster(display_name, info["slug"], info["url"])

  if idx < total_teams:
    pause = random.uniform(2.5, 4.5)
    time.sleep(pause)

print("\nاكتمل استخراج ملفات الفرق الـ 20 بنجاح.")


--- [1/20] ---

[+] جارٍ معالجة: Wadi Degla FC...
[✓] تم حفظ 27 لاعب في: wadi_degla_players.csv

--- [2/20] ---

[+] جارٍ معالجة: Modern Sport Club...
[-] لم يتم العثور على جدول اللاعبين لـ Modern Sport Club

--- [3/20] ---

[+] جارٍ معالجة: Bank El Ahly...
[✓] تم حفظ 31 لاعب في: bank_el_ahly_players.csv

--- [4/20] ---

[+] جارٍ معالجة: Cleopatra FC...
[✓] تم حفظ 20 لاعب في: cleopatra_fc_players.csv

--- [5/20] ---

[+] جارٍ معالجة: Zed FC...
[✓] تم حفظ 3 لاعب في: zed_fc_players.csv

--- [6/20] ---

[+] جارٍ معالجة: Pyramids FC...
[-] لم يتم العثور على جدول اللاعبين لـ Pyramids FC

--- [7/20] ---

[+] جارٍ معالجة: Abo Qir Fertilizers...
[-] لم يتم العثور على جدول اللاعبين لـ Abo Qir Fertilizers

--- [8/20] ---

[+] جارٍ معالجة: Qanah FC...
[-] لم يتم العثور على جدول اللاعبين لـ Qanah FC

--- [9/20] ---

[+] جارٍ معالجة: Smouha SC...
[✓] تم حفظ 32 لاعب في: smouha_sc_players.csv

--- [10/20] ---

[+] جارٍ معالجة: El Gouna FC...
[✓] تم حفظ 3 لاعب في: el_gouna_fc_players.csv

--- [11/20]